<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">《从零构建大语言模型》（Build a Large Language Model From Scratch）</a> 一书的配套代码，作者 <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 为指令数据集创建「被动语态」条目

- 本 notebook 使用 OpenAI 的 GPT-4 为指令数据集创建「被动语态」条目，示例如下

```python
{  
   'instruction': 'Identify the verb in the following sentence',
   'input': 'The cat sleeps on the couch.',
   'output': 'The verb in the sentence is "sleeps."',
   'output_2': 'The sentence is "sleeps."'   #  <---- 新创建的条目
}  
```

In [ ]:
# pip install -r requirements-extra.txt

In [ ]:
from importlib.metadata import version

pkgs = ["openai",  # OpenAI API
        "tqdm",    # 进度条
       ]

for p in pkgs:
    print(f"{p} version: {version(p)}")

&nbsp;
## 1. 测试 OpenAI API

- 首先，让我们测试 OpenAI API 是否已正确配置
- 如果还没有账户，需要在 https://platform.openai.com/ 注册
- 注意：由于 GPT-4 API 并非免费，你还需要向账户充值（参见 https://platform.openai.com/settings/organization/billing/overview）
- 使用本 notebook 中的代码创建约 200 条被动语态条目，费用约为 $0.13（13 美分）

- 首先，我们需要提供 OpenAI API 密钥，可在 https://platform.openai.com/api-keys 获取
- 请勿与任何人分享该密钥
- 将此密钥（`"sk-..."`）添加到本文件夹中的 `config.json` 文件

In [ ]:
import json
from openai import OpenAI

# 从 JSON 文件加载 API 密钥。
# 请确保将 "sk-..." 替换为你从 https://platform.openai.com/api-keys 获取的实际 API 密钥
with open("config.json", "r") as config_file:
    config = json.load(config_file)
    api_key = config["OPENAI_API_KEY"]

client = OpenAI(api_key=api_key)

- 首先，让我们用一个简单示例测试 API，确保其按预期工作：

In [ ]:
def run_chatgpt(prompt, client, model="gpt-4-turbo"):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
    )
    return response.choices[0].message.content


# 准备输入
sentence = "I ate breakfast"
prompt = f"Convert the following sentence to passive voice: '{sentence}'"
run_chatgpt(prompt, client)

&nbsp;
## 2. 创建 JSON 条目

- 接下来，我们加载要修改的文件：

In [ ]:
import json

json_file = "instruction-examples.json"

with open(json_file, "r") as file:
    json_data = json.load(file)
    
print("条目数量:", len(json_data))

- 我们先在一个小样本上试用 OpenAI Chat API，确保其正常工作：

In [ ]:
for entry in json_data[:5]:
    text = entry["output"]
    prompt = f"Without adding any response or explanation, convert the following text to passive voice: {text}"
    
    print("\n输入:")
    print(">>", text)
    print("\n输出:")
    print(">>", run_chatgpt(prompt, client))
    print("\n-------------------------")

- 现在扩展代码，将生成的条目添加到 `json_data` 中，并添加进度条：

In [ ]:
from tqdm import tqdm  # 进度条工具


for i, entry in tqdm(enumerate(json_data[:5]), total=len(json_data[:5])):
    text = entry["output"]
    prompt = f"Without adding any response or explanation, convert the following text to passive voice: {text}"
    json_data[i]["output_2"] = run_chatgpt(prompt, client)

- 再次确认新条目（`"output_2"`）是否正常

In [ ]:
json_data[0]

- 最后，如果以上内容看起来都正常，则对整个 JSON 数据集运行被动语态转换（大约需要 3 分钟）：

In [ ]:
for i, entry in tqdm(enumerate(json_data), total=len(json_data)):
    text = entry["output"]
    prompt = f"Without adding any response or explanation, convert the following text to passive voice: {text}"
    json_data[i]["output_2"] = run_chatgpt(prompt, client)

- 转换完成后，我们保存文件：

In [ ]:
new_json_file = json_file.replace(".json", "-modified.json")


with open(new_json_file, "w") as file:
    json.dump(json_data, file, indent=4)  # 使用 "indent" 进行美化输出